In [50]:
# IMDB data  sentiment analysis 
import pandas as pd
import numpy as np
df=pd.read_csv(r"C:\Users\sagar\OneDrive\Desktop\ai_ml\dl\RNN\IMDB Dataset.csv")

In [51]:
df.shape

(50000, 2)

In [52]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [53]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [54]:
df["sentiment"].nunique()

2

In [55]:
df.drop_duplicates(inplace=True)

In [56]:
df.shape

(49582, 2)

# preprocessing


### 1. converting lower case

In [57]:
df["review"]=df["review"].str.lower()

### 2. Removing URl

In [58]:
import re
sample_text="abc my name is sagar,abc"
new_text=re.sub("abc","xyz",sample_text)
new_text

'xyz my name is sagar,xyz'

In [59]:
def remove_url(text):
    text=re.sub(r"https\S+","",text)# pater replace strings
    return text
df["review"]=df["review"].apply(remove_url)

### 3. Remove panctuation

In [60]:
def remove_pan(text):
    text=re.sub(r"[^\w\s]","",text) # a-z , 0-9 \s
    return text
df["review"]=df["review"].apply(remove_pan)

### 4.Removing HTML tags

In [61]:
def remove_html(text):
    text=re.sub(r"<.*?>","",text) # a-z , 0-9 \s
    return text
df["review"]=df["review"].apply(remove_html)

### 5.Remove the stopword

In [71]:
!pip install nltk
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sagar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sagar\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sagar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [72]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [73]:
sample="sagar yadav is my fried"
token=word_tokenize(sample)

In [75]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text        

In [76]:
df["review"]=df["review"].apply(remove_stopwords)

In [77]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6.stemming

In [78]:
# porter steamming
from nltk.stem import PorterStemmer


In [80]:
def steamming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(steamming)    

### 7.encoding

In [82]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

### 8.vectorization

In [87]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000)
x=vectorizer.fit_transform(df["review"])
y=df["sentiment"]

### Data sets and data loader


In [95]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42
)

In [99]:
import torch 
from torch.utils.data import TensorDataset , DataLoader


array([[0.21337052, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(39665, 5000))

In [100]:


train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float(),
    
    
)

test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float(),
    
    
)

In [101]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
text_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [102]:
import torch.nn as nn
import torch.optim as optim

In [103]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [105]:
input_size = x_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [106]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.24660252034664154
epoch = 2/10 and loss = 0.2601778507232666
epoch = 3/10 and loss = 0.14827705919742584
epoch = 4/10 and loss = 0.2436271458864212
epoch = 5/10 and loss = 0.3358438313007355
epoch = 6/10 and loss = 0.16410502791404724
epoch = 7/10 and loss = 0.229477196931839
epoch = 8/10 and loss = 0.13193398714065552
epoch = 9/10 and loss = 0.15407203137874603
epoch = 10/10 and loss = 0.15348942577838898


In [108]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in text_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.70132096400121
